In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the project root directory by looking for a `pyproject.toml` file.

    Args:
        start (Path | None): Optional start path

    Returns:
        Path: The project root directory

    Raises:
        RuntimeError: If the project root cannot be found
    """
    current = (start or Path.cwd()).resolve()

    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent

    raise RuntimeError("Could not locate project root")


PROJECT_ROOT = find_project_root()

SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SOURCE_DIR}")

Project root: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab
Source directory: /Users/brenoingwersensantos/Documents/GitHub/vrptw-lab/src


In [2]:
from pathlib import Path

import pandas as pd

In [3]:
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize the column names to snake case and remove periods.
    """
    df.columns = [
        col.strip().lower().replace(" ", "_").replace(".", "") for col in df.columns
    ]
    return df


def load_solomon_data(path: Path):
    """
    Load Solomon data from a CSV file.
    """
    df = _normalize_columns(pd.read_csv(path))
    return df

In [4]:
name = "solomon"
instance_file = "C1/C101.csv"

In [5]:
path = Path(f"../data/{name}/{instance_file}")
df = load_solomon_data(path)
print(df.shape)
df.head()

(101, 7)


,cust_no,xcoord,ycoord,demand,ready_time,due_date,service_time
0,1,40,50,0,0,1236,0
1,2,45,68,10,912,967,90
2,3,45,70,30,825,870,90
3,4,42,66,10,65,146,90
4,5,42,68,10,727,782,90


In [6]:
from vrptw.instance import VRPTWInstance
from vrptw.problem import VRPTWProblem
from vrptw.solver import VRPTWSolver
from vrptw.solver_config import SolverConfig

instance = VRPTWInstance.from_df(df)
problem = VRPTWProblem(instance=instance, max_trucks=20, truck_capacity=200)
config = SolverConfig(max_time_in_seconds=90, log_level="DEBUG")
solver = VRPTWSolver(problem, config)

2026-09-14 00:25:36.280 | DEBUG    | vrptw.variables:__init__:65 - Creating the model variables...
2026-09-14 00:25:36.281 | DEBUG    | vrptw.variables:__init__:66 - Creating the arc variables...
2026-09-14 00:25:36.314 | DEBUG    | vrptw.variables:__init__:71 - Creating the node load variables...
2026-09-14 00:25:36.315 | DEBUG    | vrptw.variables:__init__:78 - Creating the node start time variables...
2026-09-14 00:25:36.315 | INFO     | vrptw.variables:__init__:89 - Model variables created: nodes=101, arcs=10100
2026-09-14 00:25:36.316 | DEBUG    | vrptw.solver:_add_constraints:134 - Adding constraints to the model...
2026-09-14 00:25:36.316 | DEBUG    | vrptw.solver:_add_circuit_constraint:153 - Constraint: ensure that each node is visited (multiple circuits allowed).
2026-09-14 00:25:36.324 | DEBUG    | vrptw.solver:_add_max_trucks_constraint:167 - Constraint: limit the number of trucks to 20.
2026-09-14 00:25:36.324 | DEBUG    | vrptw.solver:_add_load_constraint:181 - Constraint

In [7]:
result = solver.solve()

2026-09-14 00:25:36.478 | INFO     | vrptw.log:log_section:70 - ── Solving VRPTW problem ───────────────────────────────────
2026-09-14 00:25:36.479 | INFO     | vrptw.log:log_section:70 - ── Stage 1: Minimize trucks ────────────────────────────────
2026-09-14 00:25:36.479 | DEBUG    | vrptw.solver:_add_truck_minimization_objective:225 - Objective: minimize the number of trucks used
2026-09-14 00:25:36.959 | DEBUG    | vrptw.callback:on_solution_callback:56 - [Stage 1] objective improved: 10
2026-09-14 00:25:37.018 | INFO     | vrptw.callback:log_summary:67 - Stage 1 search: 1 improving solutions, objective 10 → 10
2026-09-14 00:25:37.018 | INFO     | vrptw.log:log_stage_complete:92 - Stage 1 complete: status=OPTIMAL, runtime=0.53s, trucks=10
2026-09-14 00:25:37.024 | DEBUG    | vrptw.solver:_fix_truck_count:244 - Constraint: fix the number of trucks to 10.
2026-09-14 00:25:37.025 | DEBUG    | vrptw.solver:_set_solution_as_hint:256 - Setting the current solution as hint for the problem

In [8]:
print(f"Status: {result.status_name}")
print(f"Trucks: {result.n_trucks}")
print(f"Runtime: {result.runtime_seconds:.2f}s")
print(f"Solver status: {solver.status_name}")

Status: OPTIMAL
Trucks: 10
Runtime: 1.24s
Solver status: OPTIMAL


In [9]:
stops_df = result.solution.to_stops_df(instance)
stops_df.head()

,truck_id,sequence,cust_no_from,cust_no_to
0,1,1,1,6
1,1,2,6,4
2,1,3,4,8
3,1,4,8,9
4,1,5,9,11
